# 52b · Bộ mã hoá EHR có phải nút thắt không — và từ vựng vi khuẩn có cấu trúc không

`46` đo `tok 0.1164` vs `XGB 0.1241` ⇒ bộ mã hoá kém `−0.0077`. Suy tỉ lệ thì chuyển sang fusion chỉ
còn `~+0.0005` — **dưới sàn nhiễu `0.0007`**. Notebook này thay suy luận đó bằng phép đo.

## 12 arm, **một** backbone forward dùng chung

Tham số các arm **rời nhau** ⇒ `d(L1+…+L12)/dθ_i = dLi/dθ_i`: huấn luyện chung một vòng lặp là
**tương đương toán học tuyệt đối** với 12 lần chạy riêng (`AMENDMENTS` ①). `38c` đã chạy 12 arm ở
~26 ph/epoch.

```
mldec                goc toa do, KHONG EHR
t1_col               EHRColTokBag(217 cot) -> Tier1        <- duong EHR hien tai
t1_col_zero          doi chung HANG SO khop tham so

t1_xgb               ColTok(40 du doan XGB OOF) -> Tier1
t1_xgb_zero          doi chung HANG SO
t1_both              ColTokBag(217 + 40)

t1_col_pre           t1_col nhung bo ma hoa DA PRE-TRAIN bang nhan (EHR-only)

t1_col_genus         MicroBag chia se theo CHI (tu dau tien cua org_name)
t1_col_genus_shuf    gan chi NGAU NHIEN, giu kich thuoc nhom
t1_col_sem           MicroBag = P @ sapbert([CLS]) + r_v
t1_col_sem_shuf      hoan vi TEN <-> id truoc khi nhung
```

## Năm câu hỏi

| tương phản | hỏi gì | đọc thế nào |
|---|---|---|
| `G(xgb) − G(col)` | bộ mã hoá có phải nút thắt | `> +0.005` ⇒ phải; `≈ 0` ⇒ trần nằm ở **nguồn nhãn** |
| `t1_col_pre` vs `t1_xgb` | do **giám sát** hay do **kiến trúc** | `pre ≈ xgb` ⇒ giám sát (đòn bẩy `B2e`, MedFuse `+0.014`) |
| `t1_both` vs `max` | cây và token-cột **cộng** hay **trùng** | `both > cả hai` ⇒ cây bắt được **tương tác** mà `T_j = b_j + x_j·W_j` (tuyến tính) không tạo nổi |
| `genus`/`sem` − `_shuf` | từ vựng vi khuẩn có cấu trúc dùng được không | đối chứng hoán vị tách **cấu trúc thật** khỏi **chỉ thêm tham số** |

## Hai luật đọc khoá trước, từ chính `52a`

**Tail của `t1_xgb` là phản chứng, không phải thành tích.** `52a` đo bội số sàn của 5 nhãn hiếm nhất:
`Azygos Lobe 0.87×` · `Clavicle Fracture 1.50×` · `Pleural Other 1.37×` · `Lobar Atelectasis 1.81×`.
4–5 cột trong 40 mang **gần như zero thông tin** (109–176 ca dương ⇒ ~87–141 ca trong mỗi model
fold). Nếu `t1_xgb` nhấc tail đáng kể ⇒ **nghi lỗi**, không phải kết quả.

**`G_ehr` lần đầu được đo nội bộ nhất quán.** Bốn lần trước ở bốn notebook khác nhau, khác `E`,
khác tập arm: `38c +0.0071 · 38d +0.0040 · 44b +0.0051 · 45 +0.0072` — tán `0.0032`, gấp 4.5× sàn
nhiễu. Ở đây mọi arm cùng vòng lặp, cùng seed, cùng `E`.

## C0 · Config

In [ ]:
import os, json, math, time, random, tarfile, io as _io, re, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r2'
RES52B  = os.path.join(OUT_DIR, 'stage4_nb52b', RUN_TAG)
CKPT    = os.path.join(RES52B, 'ckpt')
P_OOF   = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')     # tu 52a

# KHONG bao gio va `builtins.open` (de quy vo han + chet worker DataLoader).
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG: mot epoch ~26 phut khong ghi gi la du. Goi TRUOC moi cum ghi.'''
    os.makedirs(RES52B, exist_ok=True); os.makedirs(CKPT, exist_ok=True); return RES52B
ensure_res()
json.dump({'run_tag': RUN_TAG, 'notebook': '52b'}, open(os.path.join(RES52B, '_run_info.json'), 'w'))
open(os.path.join(CKPT, '_keep'), 'w').write('keep')
LOCAL = '/content/cxrlt_518'

BB_RUN = 'a2_E3'
SEEDS  = (42,)
SEED, SIZE, EPOCHS = 42, 518, 10
BATCH, ACCUM = 64, 4          # OOM voi 12 arm -> ha 32x8, giu EFF 256
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
W_FRONTAL, W_LATERAL = 0.7, 0.3
NW = min(8, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
PRE_EPOCHS, PRE_BS = 60, 512  # pre-train bo ma hoa EHR-only (C5) — nb46: ~5 phut tren GPU
SAP = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0.'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC  = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V_ = ((pd.read_parquet(P_VIEW)[['dicom_id','view_group','use']]
         .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
      else ORDER[['dicom_id','view_group']].assign(use=lambda d: d.view_group != 'unknown'))

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL       # CUNG cach cat nhu 35/37/38*/44*/45/46/52a
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA = np.where(TF_)[0], np.where(SL)[0], np.where(VA)[0]
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'backbone `ft` = 38a/{BB_RUN} | seeds {SEEDS} | E={EPOCHS} | BATCH {BATCH}x{ACCUM}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
377,110 anh | khop 219,785 | chon 39,086 | val 39,293
backbone `ft` = 38a/a2_E3 | seeds (42,) | E=10 | BATCH 64x4


## C1 · EHR + túi vi khuẩn + **đặc trưng XGB** — verbatim `45`/`46`, cộng `xgb_oof.npy`

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = _mat(COLS)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468, va `ColTok` dua thang
# x_j vao token nen score ~ M^2 -> softmax bao hoa -> NaN.
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_MICRO_RE = re.compile(r'^micro_org_')
_keep = [i for i, c in enumerate(COLS) if not _MICRO_RE.match(c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]
print(f'EHR_COL {EHR_COL.shape} ({int(_b.sum())} cot nhi phan truoc khi bo micro one-hot)')

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
NAME = _VOC.get('id_to_name', {})
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b
assert (MIDS[MCNT == 0] == PAD_ID).all() and (MIDS[MCNT > 0] != PAD_ID).all()
print(f'tui vi khuan: V={V_MIC-1} (+1 PAD) | dai toi da {MIC_L} | '
      f'anh co >=1 vi khuan {(MCNT>0).any(1).mean():.1%}')

# --- dac trung XGB out-of-fold tu 52a ---
XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C), f'xgb_oof {XGF.shape}, cho {(len(ORDER), C)}'
assert np.isfinite(XGF).all(), 'xgb_oof co gia tri khong huu han'
# logit hoa roi z-score bang thong ke LAT KHOP — xac suat tho lech nang ve 0 (prevalence 6.2%)
_lg = np.log(np.clip(XGF, 1e-6, 1 - 1e-6) / (1 - np.clip(XGF, 1e-6, 1 - 1e-6)))
_xm = _lg[TF_].mean(0, keepdims=True); _xs = _lg[TF_].std(0, keepdims=True) + 1e-6
XGB_F = np.clip((_lg - _xm) / _xs, -EHR_CLIP, EHR_CLIP).astype('float32')
print(f'XGB_F {XGB_F.shape} (logit, z-score bang thong ke LAT KHOP)')
# KHONG co arm `t1_xgb_perm`. Hoan vi 40 cot la doi chung RONG o kien truc nay: `ColTok` hoc
# `W_j`,`b_j` rieng tung chi so cot TU DAU; attention tren token EHR BAT BIEN HOAN VI (khong
# positional encoding); va query lop chi attend tren PATCH, khong tren token EHR. Nen hoan vi cot
# chi doi xem hang khoi tao nao ghep voi dac trung nao => NHIEU SEED.
# He qua TOT: kien truc KHONG the khai thac su giong hang, nen `G(xgb) - G(col)` chi co the noi ve
# THONG TIN/BIEU DIEN. Cau hoi giong hang tra loi o C14 — hau ky, 0 giay GPU.
BOTH_F = np.concatenate([EHR_COL, XGB_F], 1)
print(f'BOTH_F {BOTH_F.shape} = {EHR_COL.shape[1]} cot EHR + {C} cot XGB')

EHR_COL (377110, 217) (58 cot nhi phan truoc khi bo micro one-hot)
tui vi khuan: V=305 (+1 PAD) | dai toi da 10 | anh co >=1 vi khuan 23.5%
XGB_F (377110, 40) (logit, z-score bang thong ke LAT KHOP)
BOTH_F (377110, 257) = 217 cot EHR + 40 cot XGB


## C2 · Cấu trúc từ vựng vi khuẩn — **chi** và **ngữ nghĩa**

`genus` = từ đầu tiên của `org_name`, lấy từ chính chuỗi, **không tải gì**.
`sem` = `[CLS]` của SapBERT (đúng pooling mà nó được huấn luyện; mean-pooling sẽ **âm thầm** kém hơn).

Cổng láng giềng chạy **trước** khi tiêu giây GPU nào: nếu `STAPHYLOCOCCUS AUREUS` không có láng
giềng nào cùng chi thì model hoặc pooling sai. Cổng cũng in **tỉ lệ láng giềng-1 cùng chi** để so
`sem` với `genus` — nếu `sem` chỉ tái tạo phép gom theo chi thì nó thừa.

In [ ]:
_nm = [NAME.get(str(i), f'<id{i}>') for i in range(V_MIC - 1)] + ['<pad>']
GEN_S = [n.split()[0] if n and not n.startswith('<') else n for n in _nm]
_uq = sorted(set(GEN_S)); GEN_OF = np.array([_uq.index(g) for g in GEN_S], dtype='int64')
N_GEN = len(_uq)
_messy = [n for n in _nm[:-1] if (',' in n or '(' in n or n.split()[0] in
          ('GRAM', 'PRESUMPTIVE', 'PROBABLE', 'YEAST', 'MOLD', 'BETA', 'ALPHA'))]
print(f'{V_MIC-1} ten -> {N_GEN} chi (ke ca <pad>) | ten "ban": {len(_messy)}')
print('  chi lon nhat: ' + ' '.join(f'{g}:{int((GEN_OF==_uq.index(g)).sum())}'
      for g, _ in pd.Series(GEN_S[:-1]).value_counts().head(8).items()))
if _messy: print(f'  vi du ten ban: {_messy[:6]}')
_share = float((pd.Series(GEN_S[:-1]).value_counts() > 1).sum())
print(f'  {int(_share)} chi co >1 loai -> phep gom theo chi CHIA SE duoc; '
      f'{N_GEN-1-int(_share)} chi chi co 1 loai (khong chia se gi)')
if N_GEN - 1 > 0.8 * (V_MIC - 1):
    print('  !! gan nhu moi ten mot chi -> arm `genus` se GAN BANG `t1_col`. Van chay, nhung')
    print('     dung doc ket qua cua no nhu bang chung chong lai cau truc phan loai.')

# --- SapBERT: TUY CHON. Khong tai duoc thi bo hai arm `sem`, phan con lai van chay. ---
P_SEM = os.path.join(RES52B, 'sem_emb.npy')
SEM = None
if os.path.exists(P_SEM):
    SEM = np.load(P_SEM); print(f'\nnap lai sem_emb {SEM.shape}')
else:
    try:
        from transformers import AutoTokenizer, AutoModel as _AM
        _tk = AutoTokenizer.from_pretrained(SAP); _bm = _AM.from_pretrained(SAP).to(dev).eval()
        _out = []
        with torch.no_grad():
            for i in range(0, len(_nm), 64):
                b = _tk(_nm[i:i+64], padding=True, truncation=True, max_length=32,
                        return_tensors='pt').to(dev)
                _out.append(_bm(**b).last_hidden_state[:, 0].float().cpu().numpy())  # [CLS]
        SEM = np.concatenate(_out).astype('float32'); SEM[-1] = 0.0                  # <pad> = 0
        del _bm; torch.cuda.empty_cache()
        ensure_res(); np.save(P_SEM, SEM); print(f'\nsem_emb {SEM.shape} -> {P_SEM}')
    except Exception as e:
        print(f'\n!! khong nhung duoc SapBERT ({type(e).__name__}: {e}) -> BO hai arm `sem`')

USE_SEM = SEM is not None
if USE_SEM:
    _z = SEM[:-1] / (np.linalg.norm(SEM[:-1], axis=1, keepdims=True) + 1e-9)
    _S = _z @ _z.T; np.fill_diagonal(_S, -9)
    _nn = _S.argmax(1)
    _hit = float(np.mean([GEN_OF[i] == GEN_OF[_nn[i]] for i in range(len(_z))]))
    print(f'\ncong sem: lang gieng-1 CUNG CHI {_hit:.1%} (ngau nhien ~ {1.0/N_GEN:.1%})')
    for i in range(min(4, len(_z))):
        print(f'    {_nm[i][:34]:<36} -> {_nm[_nn[i]][:34]}')
    assert _hit > 3.0 / N_GEN, (f'lang gieng-1 cung chi chi {_hit:.1%} — SapBERT khong tach duoc '
                                'ten vi khuan. Sai model hoac sai pooling (phai la [CLS]).')
    print(f'  {_hit:.1%} << 100% nghia la `sem` gom KHAC `genus` — hai arm khong trung nhau.')

305 ten -> 132 chi (ke ca <pad>) | ten "ban": 56
  chi lon nhat: POSITIVE:16 STREPTOCOCCUS:12 CANDIDA:12 MYCOBACTERIUM:11 GRAM:9 ASPERGILLUS:8 PSEUDOMONAS:8 SALMONELLA:8
  vi du ten ban: ['YEAST', 'STAPHYLOCOCCUS, COAGULASE NEGATIVE', 'GRAM NEGATIVE ROD(S)', 'GRAM POSITIVE BACTERIA', 'HAEMOPHILUS INFLUENZAE, BETA-LACTAMASE NEGATIVE', 'CORYNEBACTERIUM SPECIES (DIPHTHEROIDS)']
  55 chi co >1 loai -> phep gom theo chi CHIA SE duoc; 76 chi chi co 1 loai (khong chia se gi)


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


sem_emb (306, 768) -> /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb52b/r2/sem_emb.npy

cong sem: lang gieng-1 CUNG CHI 61.6% (ngau nhien ~ 0.8%)
    <unk>                                -> 2ND ISOLATE
    YEAST                                -> SACCHAROMYCES CEREVISIAE
    STAPH AUREUS COAG +                  -> POSITIVE FOR METHICILLIN RESISTANT
    ESCHERICHIA COLI                     -> ENTEROBACTER SPECIES
  61.6% << 100% nghia la `sem` gom KHAC `genus` — hai arm khong trung nhau.


## C3 · Ảnh + backbone `ft` đông cứng

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_n_local = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n_local:,} / {len(ORDER):,}')
if _n_local < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        name, blob = pl; dst = os.path.join(LOCAL, os.path.basename(name))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n_local = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n_local >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n_local:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
print(f'BACKBONE `ft`: 38a/{BB_RUN} | lech so voi pretrain {_d:.3f}. CHI lay `bb`, KHONG lay `head`.')

class DS(Dataset):
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        j = i if s.perm is None else int(s.perm[k])
        return dict(image=s.tf(Image.merge('RGB', (im, im, im))), y=torch.from_numpy(YM[i]),
                    row=i, erow=j, pos=k)

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

@torch.no_grad()
def fwd(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone(pixel_values=img).last_hidden_state[:, 1:].float()

_b0 = next(iter(loader(I_VA[:8], eval_tf, False, 8, nw=0)))
N_TOK = fwd(_b0['image'].to(dev)).shape[1]
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'patch {N_TOK} token')

anh local: 377,110 / 377,110


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft`: 38a/a2_E3 | lech so voi pretrain 0.021. CHI lay `bb`, KHONG lay `head`.
patch 1369 token


## C4 · Kiến trúc — `MLDecoder`/`Tier1`/`ASL` verbatim `45`, `MicroBag` thêm hai chế độ

`MicroBag` giờ có ba chế độ sinh bảng nhúng `[V, d]`:

```
free   W tu do                        vi khuan hiem hoc tu vai chuc vi du -> nhieu
genus  g[chi(v)] + r_v,  r_v init 0   loai hiem thua huong toan bo tu cac loai cung chi
sem    emb0 @ P + r_v,   r_v init 0   P HOC CHUNG cho moi loai; bert DONG CUNG
                                      -> loai hiem co ZERO tham so rieng
```
`r_v` init 0 ⇒ arm khởi động **đúng bằng prior thuần**, giữ kỷ luật zero-init của dự án.

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class Tier1(nn.Module):
    '''B2a — Q = 1369 PATCH, K/V = token EHR. softmax CHI tren token EHR nen EHR khong the bi
       1369 patch nhan chim (co che that bai cua 06b H6). `pr` zero-init; `ln_a` chuan hoa DAU VAO
       cua phep chieu (ban va 38d — thieu no thi `t1_zero` sup tu 0.2416 xuong 0.0681).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))
    def attn(s, patch, etok):
        '''CHI de chan doan (C13). `forward` dung need_weights=False cho duong SDPA hop nhat chay,
           khong hien thuc hoa ma tran [1369, 219]. O day co y hien thuc hoa, tren mau nho.'''
        _, w = s.at(s.n(patch), etok, etok, need_weights=True, average_attn_weights=True)
        return w

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = 1369 patch, co the da bi Tier1 dieu bien.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class MicroBag(nn.Module):
    '''`const=True`: doi chung khop tham so — GIU nguyen bang nhung ep moi chi so ve 0 va trong so
       ve 1 => mot VECTOR HANG SO hoc duoc, khong doc benh nhan (mau H4z/H6z cua 06b).
       MOT chi so, khong phai MIC_L ban sao: tui that trung binh 1.7 vi khuan.'''
    def __init__(s, V, d, const=False, mode='free', grp=None, n_grp=0, emb0=None):
        super().__init__()
        s.V, s.d, s.const, s.mode = V, d, const, mode
        if mode != 'free':
            s.r = nn.Parameter(torch.zeros(V, d))        # du hoc rieng, INIT 0
        if mode == 'free':
            s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        elif mode == 'genus':
            s.register_buffer('grp', torch.as_tensor(grp, dtype=torch.long))
            s.g = nn.Parameter(torch.randn(n_grp, d) * d ** -0.5)
        elif mode == 'sem':
            s.register_buffer('e0', torch.as_tensor(emb0, dtype=torch.float32))
            s.P = nn.Parameter(torch.randn(emb0.shape[1], d) * emb0.shape[1] ** -0.5)
        else: raise ValueError(mode)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))   # khop dang cua EHRColTok
    def table(s):
        if s.mode == 'free':  return s.w
        if s.mode == 'genus': return s.g[s.grp] + s.r
        return s.e0 @ s.P + s.r
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w   = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.table(), per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''217 token COT (FT-Transformer: T_j = b_j + x_j*W_j) + 1 token TONG HOP (MLP nhin CA 217
       cot cung luc) + 1 token BAG vi khuan (tuy chon).

       Chuoi nay CHUA k1 nhu TRUONG HOP RIENG — attention chi can don het khoi luong len token tong
       hop la tai lap duoc k1. `EHRTok` theo HO (K=9) khong co tinh chat do, va do la ly do no THUA
       k1 o v1 cua 37. (`EHRTok`/`famsplit` la DI SAN 37/38b, KHONG dung o day.)'''
    def __init__(s, d_in, d=256, const=False, V=None, **mb):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)
        s.mb = MicroBag(V, d, const, **mb) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d)
            s.cln = nn.LayerNorm(d) if const else nn.Identity()   # duong that da co s.ln
    def forward(s, e, mids=None, mcnt=None):
        if s.const: base = s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        else:
            col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
            base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class AttnPool(nn.Module):
    '''Gop bang MOT query HOC DUOC — GIU dung DANG ma `Tier1` dung o downstream (mot truy van
       attend tren token EHR, softmax tren TUNG token).

       Vi sao KHONG `mean(token)`: trung binh day bo ma hoa rang CHI trung binh moi quan trong,
       nen no co the lam 219 token du thua lan nhau; roi ta cay no vao `Tier1` — noi doc TUNG
       token. Lech muc tieu do lam `t1_col_pre` khong con doc duoc: no thua thi khong phan biet
       duoc "giam sat vo dung" voi "giam sat SAI DANG".

       Vi sao KHONG 40 query DONG CUNG kieu ML-Decoder: `46` do `mldec` EHR-only = 0.1049 — TE
       NHAT trong bon dau doc, va bien do seed 0.0073, gap 12x cac dau doc kia.'''
    def __init__(s, d, C, nhead=8):
        super().__init__()
        s.q = nn.Parameter(torch.randn(1, 1, d) * d ** -0.5)
        s.at = nn.MultiheadAttention(d, nhead, batch_first=True)
        s.ln = nn.LayerNorm(d); s.fc = nn.Linear(d, C)
    def forward(s, etok):
        h, _ = s.at(s.q.expand(etok.shape[0], -1, -1), etok, etok, need_weights=False)
        return s.fc(s.ln(h.squeeze(1)))

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

kien truc san sang


## C5 · Pre-train bộ mã hoá EHR-only *(cho `t1_col_pre`)*

`46` **không lưu** state dict, nên làm lại ở đây — và như vậy bộ mã hoá dùng **đúng** ma trận đặc
trưng, **đúng** lát cắt, **đúng** lớp `ColTokBag` của notebook này. `46` đo `mldec` chạy 60 epoch
trên GPU trong `5.2` phút; `tok` nhẹ hơn.

Cổng: `mAP` EHR-only phải rơi vào vùng `46` đã đo (`tok 0.1164`, biên độ seed `0.0006`).

In [ ]:
P_PRE = os.path.join(RES52B, 'enc_pre.pt')
MOC46_TOK = 0.1164      # `tok` cua 46 dung MEAN-pool; o day la ATTENTION pool nen co the khac
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o

if os.path.exists(P_PRE):
    print(f'nap lai {P_PRE} (mAP EHR-only {torch.load(P_PRE, weights_only=False)["mAP"]:.4f})')
else:
    torch.manual_seed(SEED)
    enc = ColTokBag(EHR_COL.shape[1], D_ETOK, False, V=V_MIC, mode='free').to(dev)
    hd  = AttnPool(D_ETOK, C).to(dev)      # KHONG mean(token) — xem docstring AttnPool
    crit0 = ASL(gn=4, gp=0).to(dev)
    opt0 = torch.optim.AdamW(list(enc.parameters()) + list(hd.parameters()), lr=3e-3,
                             weight_decay=1e-4)
    sch0 = torch.optim.lr_scheduler.CosineAnnealingLR(opt0, PRE_EPOCHS)
    Tt = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=dev)
    t0 = time.time()
    for ep in range(PRE_EPOCHS):
        enc.train(); hd.train()
        pm = np.random.default_rng(SEED * 1000 + ep).permutation(len(I_TF))
        for i in range(0, len(I_TF), PRE_BS):
            k = I_TF[pm[i:i+PRE_BS]]
            opt0.zero_grad(set_to_none=True)
            lg = hd(enc(Tt(EHR_COL[k]), Tt(MIDS[k], torch.long), Tt(MCNT[k])))
            crit0(lg, Tt(YM[k])).backward(); opt0.step()
        sch0.step()
        if ep == 0 or (ep + 1) % 20 == 0:
            print(f'  pre ep{ep+1:>2}/{PRE_EPOCHS}  {(time.time()-t0)/60:.1f} ph', flush=True)
    enc.eval(); hd.eval(); ps = []
    with torch.no_grad():
        for i in range(0, len(I_VA), PRE_BS):
            k = I_VA[i:i+PRE_BS]
            ps.append(torch.sigmoid(hd(enc(Tt(EHR_COL[k]), Tt(MIDS[k], torch.long),
                                           Tt(MCNT[k])))).cpu().numpy())
    _mp = float(np.nanmean(ap_cols(YM[VA], np.concatenate(ps))))
    ensure_res(); torch.save(dict(enc=enc.state_dict(), mAP=_mp), P_PRE)
    print(f'pre-train xong | mAP EHR-only {_mp:.4f} | moc 46 `tok` {MOC46_TOK:.4f} | '
          f'{(time.time()-t0)/60:.1f} ph')
    assert abs(_mp - MOC46_TOK) < 0.02, (
        f'lech {abs(_mp-MOC46_TOK):.4f} so voi `tok` cua 46 -> bo ma hoa pre-train khong o cung '
        'vung, dung dung cho t1_col_pre. Nguong 0.02 chu khong 0.01 vi dau doc o day la ATTENTION '
        'POOL chu khong phai mean-pool: CAO hon la dau TOT, THAP hon nhieu moi la van de.')
    del enc, hd, opt0, sch0; torch.cuda.empty_cache()

  pre ep 1/60  0.1 ph
  pre ep20/60  1.3 ph
  pre ep40/60  2.6 ph
  pre ep60/60  3.9 ph
pre-train xong | mAP EHR-only 0.1069 | moc 46 `tok` 0.1164 | 3.9 ph


## C6 · 12 arm + cổng khoá trước

In [ ]:
ARMS = {
  'mldec':             dict(feat=None),
  't1_col':            dict(feat='col'),
  't1_col_zero':       dict(feat='col',  const=True),
  't1_xgb':            dict(feat='xgb'),
  't1_xgb_zero':       dict(feat='xgb',  const=True),
  't1_both':           dict(feat='both'),
  't1_col_pre':        dict(feat='col',  pre=True),
  't1_col_genus':      dict(feat='col',  mb='genus'),
  't1_col_genus_shuf': dict(feat='col',  mb='genus', shuf=True),
  't1_col_sem':        dict(feat='col',  mb='sem'),
  't1_col_sem_shuf':   dict(feat='col',  mb='sem',   shuf=True),
}
if not USE_SEM:
    ARMS = {k: v for k, v in ARMS.items() if 'sem' not in k}
    print('!! bo hai arm `sem` (khong nhung duoc SapBERT)')
FEAT = {'col': EHR_COL, 'xgb': XGB_F, 'both': BOTH_F}
_rs_s = np.random.default_rng(SEED + 11)
GEN_SHUF = GEN_OF[_rs_s.permutation(V_MIC)]                 # gan chi NGAU NHIEN, giu kich thuoc nhom
SEM_SHUF = SEM[_rs_s.permutation(V_MIC)] if USE_SEM else None   # hoan vi TEN <-> id

def build_arms(seed=SEED):
    M, E = nn.ModuleDict(), nn.ModuleDict()
    _pre = torch.load(P_PRE, map_location='cpu', weights_only=False)['enc'] \
           if os.path.exists(P_PRE) else None
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)          # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        if cfg['feat'] is None:
            M[tag] = MLDecoder(C, D_CXR, D_MLD); continue
        X = FEAT[cfg['feat']]; cst = cfg.get('const', False)
        mb = cfg.get('mb', 'free')
        kw = dict(mode=mb)
        if mb == 'genus':
            kw.update(grp=(GEN_SHUF if cfg.get('shuf') else GEN_OF), n_grp=N_GEN)
        elif mb == 'sem':
            kw.update(emb0=(SEM_SHUF if cfg.get('shuf') else SEM))
        E[tag] = ColTokBag(X.shape[1], D_ETOK, cst, V=V_MIC, **kw)
        if cfg.get('pre'):
            _m = E[tag].load_state_dict(_pre, strict=False)
            assert not _m.missing_keys, f'{tag}: thieu khoa khi nap pre-train: {_m.missing_keys[:5]}'
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=True)
    return M.to(dev), E.to(dev)

MOD, ETK = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + ([ETK[t]] if t in ETK else []))
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<20} {"tham so":>12}  {"dac trung":>10}')
for t, c in ARMS.items():
    print(f'  {t:<18} {NP_[t]:>12,}  {str(c["feat"]):>10}')

_e  = {k: torch.randn(4, v.shape[1], device=dev) for k, v in FEAT.items()}
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev)
_mc = torch.zeros(4, MIC_L, device=dev)
_p  = torch.randn(4, N_TOK, D_CXR, device=dev)
etok = lambda t: ETK[t](_e[ARMS[t]['feat']], _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'
# --- cong 1: Tier1 zero-init => bo tang 1 phai cho logit Y HET ---
with torch.no_grad():
    m = MOD['t1_col'].eval(); l1, _ = m(_p, etok('t1_col'))
    _k = m.t1; m.t1 = None; l2, _ = m(_p, etok('t1_col')); m.t1 = _k
    d1 = (l1 - l2).abs().max().item()
assert d1 < 1e-5, f'Tier1 KHONG zero-init: lech {d1:.2e}'
print(f'\ncong 1: Tier1 zero-init OK (lech {d1:.1e})')
# --- cong 2a: doi chung HOAN VI phai khop tham so TUYET DOI (cung kien truc, chi khac BUFFER) ---
for a, b in (('t1_col_genus','t1_col_genus_shuf'), ('t1_col_sem','t1_col_sem_shuf')):
    if a in ARMS and b in ARMS:
        assert NP_[a] == NP_[b], f'khop tham so HONG: {a} {NP_[a]:,} vs {b} {NP_[b]:,}'
        print(f'cong 2a: {a} = {b} ({NP_[a]:,} tham so) — khop TUYET DOI')
# --- cong 2b: doi chung HANG SO (`_zero`) KHONG khop tuyet doi va khong the khop:
#     che do `const` cua `ColTokBag` van tao W/b/joint (khong dung) roi THEM `c` va `cln`.
#     Do la thiet ke cua 38c/38d/45, giu nguyen de so duoc voi so cu. Huong lech la CO LOI cho
#     doi chung (no NHIEU tham so hon), nen `that - doi_chung > 0` la ket luan BAO THU. ---
for a, b in (('t1_col','t1_col_zero'), ('t1_xgb','t1_xgb_zero')):
    if a in ARMS and b in ARMS:
        assert NP_[b] >= NP_[a], f'{b} IT tham so hon {a} -> doi chung khong con bao thu'
        print(f'cong 2b: {b} {NP_[b]:,} >= {a} {NP_[a]:,} (+{NP_[b]-NP_[a]:,}) '
              f'-> `{a} - {b} > 0` la BAO THU')
# --- cong 3: `t1_xgb` co IT tham so hon `t1_col` -> mot chien thang la BAO THU ---
print(f'cong 3: t1_xgb {NP_["t1_xgb"]:,} < t1_col {NP_["t1_col"]:,} tham so '
      f'({NP_["t1_col"]-NP_["t1_xgb"]:,} it hon) -> t1_xgb thang thi ket luan la BAO THU')
assert NP_['t1_xgb'] < NP_['t1_col'], (
    f"t1_xgb ({NP_['t1_xgb']:,}) KHONG it tham so hon t1_col ({NP_['t1_col']:,}). "
    f'Voi {EHR_COL.shape[1]} cot EHR va {C} cot XGB thi bat dang thuc dao chieu, va mot chien '
    'thang cua t1_xgb khong con la ket luan bao thu.')
# --- cong 4: ba che do MicroBag PHAI sinh ra bang nhung KHAC nhau ---
with torch.no_grad():
    _tb = {t: ETK[t].mb.table().clone() for t in ARMS if t in ETK and ETK[t].mb is not None}
_ks = [t for t in ('t1_col','t1_col_genus','t1_col_sem') if t in _tb]
for i, a in enumerate(_ks):
    for b in _ks[i+1:]:
        _dd = (_tb[a] - _tb[b]).abs().max().item()
        assert _dd > 1e-4, (f'bang nhung cua `{a}` va `{b}` GIONG HET (lech {_dd:.1e}) -> '
                            '`MicroBag.table()` khong re nhanh theo `mode`')
print(f'cong 4: ba che do MicroBag sinh bang KHAC nhau OK ({_ks})')
# --- cong 5: `r_v` init 0 => arm genus/sem khoi dong DUNG BANG prior thuan ---
for t in ARMS:
    if t in ETK and ETK[t].mb is not None and ARMS[t].get('mb', 'free') != 'free':
        assert float(ETK[t].mb.r.abs().max()) == 0.0, f'{t}: r_v KHONG init 0'
print('cong 5: `r_v` init 0 o moi arm genus/sem OK')

arm                       tham so   dac trung
  mldec                 2,517,032        None
  t1_col                4,341,800         col
  t1_col_zero           4,398,120         col
  t1_xgb                4,205,864         xgb
  t1_xgb_zero           4,216,872         xgb
  t1_both               4,372,520        both
  t1_col_pre            4,341,800         col
  t1_col_genus          4,375,592         col
  t1_col_genus_shuf     4,375,592         col
  t1_col_sem            4,538,408         col
  t1_col_sem_shuf       4,538,408         col

cong 1: Tier1 zero-init OK (lech 0.0e+00)
cong 2a: t1_col_genus = t1_col_genus_shuf (4,375,592 tham so) — khop TUYET DOI
cong 2a: t1_col_sem = t1_col_sem_shuf (4,538,408 tham so) — khop TUYET DOI
cong 2b: t1_col_zero 4,398,120 >= t1_col 4,341,800 (+56,320) -> `t1_col - t1_col_zero > 0` la BAO THU
cong 2b: t1_xgb_zero 4,216,872 >= t1_xgb 4,205,864 (+11,008) -> `t1_xgb - t1_xgb_zero > 0` la BAO THU
cong 3: t1_xgb 4,205,864 < t1_col 4,341,800 tha

## C7 · Chấm điểm — `ap_cols` tự viết + gộp study, verbatim `45`

In [ ]:
from sklearn.metrics import average_precision_score
_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V_.view_group.to_numpy()[VA] == 'frontal') & V_.use.to_numpy()[VA]
_isL = (V_.view_group.to_numpy()[VA] == 'lateral') & V_.use.to_numpy()[VA]
def pool_study(p):
    out = np.zeros((len(_uid), p.shape[1]), 'float64'); ws = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(p.shape[1])], 1)
        has = cnt > 0; out[has] += w * acc[has] / cnt[has, None]; ws[has] += w
    ok = ws > 0; out[ok] /= ws[ok, None]; return out, ok
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss6 = _rs6.random((3000, C)).astype('float32')
_dd6 = float(np.nanmax(np.abs(ap_cols(_yy, _ss6) -
        np.array([average_precision_score(_yy[:, j], _ss6[:, j]) for j in range(C)]))))
assert _dd6 < 1e-6, f'ap_cols lech sklearn {_dd6:.2e}'
print(f'cong 6: ap_cols khop sklearn toi {_dd6:.2e} | val {VA.sum():,} anh -> {len(_uid):,} study')

cong 6: ap_cols khop sklearn toi 1.69e-08 | val 39,293 anh -> 23,800 study


## C8 · Huấn luyện — 12 arm, **một** backbone forward

`clip` gradient **riêng từng arm**; arm nào cho logit không hữu hạn thì **bỏ arm đó ở bước đó**,
không giết cả lần chạy. Checkpoint **mỗi epoch** ⇒ đứt mất đúng một epoch. `loss` in **trung bình
từng arm**, không phải tổng.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = {k: torch.from_numpy(v) for k, v in FEAT.items()}

def run_arms(M, E, patch, erow):
    out = {}
    for tag, cfg in ARMS.items():
        if cfg['feat'] is None: out[tag] = M[tag](patch); continue
        e = FEAT_T[cfg['feat']][erow].to(dev)
        t = E[tag](e, MIDS_T[erow].to(dev), MCNT_T[erow].to(dev))
        out[tag] = M[tag](patch, t)
    return out

@torch.no_grad()
def predict(M, E, idx, perm=None, bs=48):
    M.eval(); E.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(loader(idx, eval_tf, False, bs, perm, nw=0, drop_last=False),
                  desc='eval', leave=False):
        pt = fwd(b['image'].to(dev, non_blocking=True))
        for t, (lg, _) in run_arms(M, E, pt, b['erow']).items():
            P[t][b['pos'].numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
    _bad = {t: int(np.isnan(v).sum()) for t, v in P.items() if np.isnan(v).any()}
    assert not _bad, f'predict cho ra NaN: {_bad}'
    for t in ARMS: assert np.abs(P[t]).sum(1).min() > 0, f'{t}: hang khong duoc ghi -> `pos` sai'
    return P

P_LOG = os.path.join(RES52B, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL = loader(I_TF, train_tf, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD, ETK = build_arms(seed)
    PARAM_OF = {t: [p for m in ([MOD[t]] + ([ETK[t]] if t in ETK else []))
                    for p in m.parameters() if p.requires_grad] for t in ARMS}
    N_CLIP = {t: 0 for t in ARMS}; N_SKIP = {t: 0 for t in ARMS}; N_STEP = 0
    rest = [p for m in (MOD, ETK) for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': [p for p in rest if p.ndim > 1], 'weight_decay': WD},
                             {'params': [p for p in rest if p.ndim <= 1], 'weight_decay': 0.0}],
                            lr=LR)
    steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS; warm = int(WARMUP * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt'); ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        MOD.load_state_dict(_r['mod']); ETK.load_state_dict(_r['etk'])
        opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        MOD.train(); ETK.train()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            pt = fwd(b['image'].to(dev, non_blocking=True)); y = b['y'].to(dev)
            loss, bad = 0.0, []
            for t, (lg, _) in run_arms(MOD, ETK, pt, b['erow']).items():
                if not torch.isfinite(lg).all(): bad.append(t); N_SKIP[t] += 1; continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if bad and (sum(N_SKIP.values()) <= 3 or j % 400 == 0):
                _mx = {t: max(float(p.abs().max()) for p in PARAM_OF[t]) for t in bad}
                print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad} | '
                      + ' '.join(f'{t}:|w|max {v:.3g}' for t, v in _mx.items()))
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc: total_norm = nan cho clip_coef = nan va BOI BAN
                # toan bo tham so -> mot buoc hong lam chet vinh vien.
                for t, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()
                        N_SKIP[t] += 1
                    elif float(gn) > CLIP: N_CLIP[t] += 1
                N_STEP += 1; opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        PV = predict(MOD, ETK, I_VA); PS = predict(MOD, ETK, I_SL)
        ensure_res()
        for t, p in PV.items(): np.save(os.path.join(RES52B, f'p_{t}_{sk}_ep{ep+1}.npy'), p)
        # LUU CA lat CHON: C14 chon trong so tron TREN DAY roi bao cao tren val. Chon tren val roi
        # bao cao tren val la dung cai bay §13 cua ke hoach.
        for t, p in PS.items(): np.save(os.path.join(RES52B, f'psl_{t}_{sk}_ep{ep+1}.npy'), p)
        _e2 = {}
        for t, p in PV.items():
            ap = ap_cols(YM[VA], p)
            _e2[t] = dict(mAP_anh=float(np.nanmean(ap)), **tiers(ap),
                          sel=float(np.nanmean(ap_cols(YM[SL], PS[t]))))
        LOG[sk][str(ep)] = _e2; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), opt=opt.state_dict(),
                        sch=sch.state_dict(), n_clip=N_CLIP, n_skip=N_SKIP, n_step=N_STEP,
                        ep=ep, seed=seed), P_RES)
        print(f'{sk} ep{ep+1:>2} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph | loss ' +
              ' '.join(f'{t.replace("t1_","")}:{tot[t]/max(nb,1):.3f}' for t in ARMS))
        _g = lambda a: _e2[a]['mAP_anh'] - _e2['mldec']['mAP_anh']
        print('   VAL      ' + ' '.join(f'{t.replace("t1_",""):>10}:{_e2[t]["mAP_anh"]:.4f}'
                                        for t in ARMS))
        print('   LAT CHON ' + ' '.join(f'{t.replace("t1_",""):>10}:{_e2[t]["sel"]:.4f}'
                                        for t in ARMS))
        print('   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`')
        print(f'   G(col) {_g("t1_col"):+.4f} | G(xgb) {_g("t1_xgb"):+.4f} | '
              f'G(xgb)-G(col) {_g("t1_xgb")-_g("t1_col"):+.4f}  <- CON SO CAN DO')
        print(f'   G(both) {_g("t1_both"):+.4f} | G(pre) {_g("t1_col_pre"):+.4f}')
        print(f'   moc G_ehr cac notebook truoc: 38c +0.0071 · 38d +0.0040 · 44b +0.0051 · 45 +0.0072')
        _f = {t: N_CLIP[t]/max(N_STEP,1) for t in ARMS}
        if max(_f.values()) > 0.01:
            print('   clip: ' + ' '.join(f'{t}:{v:.0%}' for t, v in _f.items() if v > 0.01))
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO: ' + ' '.join(f'{t}:{v}' for t, v in N_SKIP.items() if v > 0)
                  + '  -> arm nao bi bo nhieu thi con so cua no KHONG doc duoc')
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), seed=seed, n_skip=N_SKIP),
               os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong')
print('tat ca seed xong')

s42 ep1/10:   0%|          | 0/3434 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s42 ep 1 | 103 img/s | 35.4 ph | loss mldec:0.043 col:0.043 col_zero:0.043 xgb:0.042 xgb_zero:0.042 both:0.041 col_pre:0.043 col_genus:0.043 col_genus_shuf:0.043 col_sem:0.042 col_sem_shuf:0.042
   VAL           mldec:0.2382        col:0.2393   col_zero:0.2409        xgb:0.2427   xgb_zero:0.2413       both:0.2438    col_pre:0.2351  col_genus:0.2424 col_genus_shuf:0.2414    col_sem:0.2409 col_sem_shuf:0.2407
   LAT CHON      mldec:0.2317        col:0.2307   col_zero:0.2322        xgb:0.2356   xgb_zero:0.2329       both:0.2348    col_pre:0.2268  col_genus:0.2348 col_genus_shuf:0.2332    col_sem:0.2332 col_sem_shuf:0.2323
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   G(col) +0.0011 | G(xgb) +0.0045 | G(xgb)-G(col) +0.0034  <- CON SO CAN DO
   G(both) +0.0056 | G(pre) -0.0031
   moc G_ehr cac notebook truoc: 38c +0.0071 · 38d +0.0040 · 44b +0.0051 · 45 +0.0072


s42 ep2/10:   0%|          | 0/3434 [00:00<?, ?it/s]

...

s42 ep 9 | 104 img/s | 35.4 ph | loss mldec:0.036 col:0.036 col_zero:0.036 xgb:0.036 xgb_zero:0.036 both:0.036 col_pre:0.035 col_genus:0.036 col_genus_shuf:0.036 col_sem:0.036 col_sem_shuf:0.036
   VAL           mldec:0.2371        col:0.2454   col_zero:0.2465        xgb:0.2479   xgb_zero:0.2408       both:0.2477    col_pre:0.2361  col_genus:0.2476 col_genus_shuf:0.2457    col_sem:0.2475 col_sem_shuf:0.2452
   LAT CHON      mldec:0.2325        col:0.2389   col_zero:0.2349        xgb:0.2398   xgb_zero:0.2349       both:0.2408    col_pre:0.2300  col_genus:0.2381 col_genus_shuf:0.2353    col_sem:0.2385 col_sem_shuf:0.2374
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   G(col) +0.0084 | G(xgb) +0.0109 | G(xgb)-G(col) +0.0025  <- CON SO CAN DO
   G(both) +0.0106 | G(pre) -0.0010
   moc G_ehr cac notebook truoc: 38c +0.0071 · 38d +0.0040 · 44b +0.0051 · 45 +0.0072


s42 ep10/10:   0%|          | 0/3434 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s42 ep10 | 104 img/s | 35.4 ph | loss mldec:0.036 col:0.036 col_zero:0.036 xgb:0.036 xgb_zero:0.036 both:0.036 col_pre:0.034 col_genus:0.036 col_genus_shuf:0.036 col_sem:0.036 col_sem_shuf:0.036
   VAL           mldec:0.2369        col:0.2453   col_zero:0.2468        xgb:0.2477   xgb_zero:0.2406       both:0.2475    col_pre:0.2361  col_genus:0.2472 col_genus_shuf:0.2454    col_sem:0.2473 col_sem_shuf:0.2450
   LAT CHON      mldec:0.2325        col:0.2388   col_zero:0.2347        xgb:0.2399   xgb_zero:0.2348       both:0.2411    col_pre:0.2302  col_genus:0.2383 col_genus_shuf:0.2353    col_sem:0.2387 col_sem_shuf:0.2376
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   G(col) +0.0084 | G(xgb) +0.0108 | G(xgb)-G(col) +0.0024  <- CON SO CAN DO
   G(both) +0.0107 | G(pre) -0.0007
   moc G_ehr cac notebook truoc: 38c +0.0071 · 38d +0.0040 · 44b +0.0051 · 45 +0.0072
s42: xong
tat ca seed xong


## C9 · Năm câu hỏi — bảng và cách đọc đã khai báo trước

In [ ]:
LOG = json.load(open(P_LOG)); EPk = str(EPOCHS - 1)
rows = [dict(seed=s, arm=t, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
        for t, v in LOG[f's{s}'][EPk].items()]
R = pd.DataFrame(rows); assert len(R), 'chua co ket qua nao'
SD = [s for s in SEEDS if EPk in LOG.get(f's{s}', {})]
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel'],
                    aggfunc='mean').reindex(list(ARMS))
piv['tham_so'] = [NP_[t] for t in ARMS]
print('=== TUYET DOI (TB qua seed) ===')
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))

g  = lambda a, s: float(R[(R.arm == a) & (R.seed == s)].mAP_anh.iloc[0])
gt = lambda a, s: float(R[(R.arm == a) & (R.seed == s)]['tail'].iloc[0])
G  = lambda a: float(np.mean([g(a, s) - g('mldec', s) for s in SD]))
Gt = lambda a: float(np.mean([gt(a, s) - gt('mldec', s) for s in SD]))
def sgn(a, b):
    v = [g(a, s) - g(b, s) for s in SD]
    return ('cung dau' if len(v) > 1 and (all(x > 0 for x in v) or all(x < 0 for x in v))
            else ('1 seed' if len(v) < 2 else 'DOI DAU'))

print('\n=== G = arm - mldec (bao gom ca DUNG LUONG) ===')
for t in ARMS:
    if t == 'mldec': continue
    print(f'  {t:<20} {G(t):+.4f}   tail {Gt(t):+.4f}')
print('\n=== NOI DUNG THUAN = arm - doi chung KHOP THAM SO ===')
for a, b in (('t1_col','t1_col_zero'), ('t1_xgb','t1_xgb_zero'),
             ('t1_col_genus','t1_col_genus_shuf'), ('t1_col_sem','t1_col_sem_shuf')):
    if a in ARMS and b in ARMS:
        v = [g(a, s) - g(b, s) for s in SD]
        print(f'  {a:<20} - {b:<20} {np.mean(v):+.4f}  ({sgn(a,b)})')

print('\n' + '=' * 74)
D = G('t1_xgb') - G('t1_col')
print(f'CAU 1  G(xgb) - G(col) = {D:+.4f}')
print(f'       G(col) {G("t1_col"):+.4f}  vs 4 notebook truoc: +0.0071/+0.0040/+0.0051/+0.0072')
print('       > +0.005  -> BO MA HOA la nut that       | ~0 -> KHONG phai, tran o NGUON NHAN')
Dr = G('t1_col_pre') - G('t1_col')
print(f'\nCAU 2  G(pre) - G(col) = {Dr:+.4f}   | G(pre) so G(xgb): {G("t1_col_pre")-G("t1_xgb"):+.4f}')
print('       pre ~ xgb -> nguyen nhan la GIAM SAT. Sua: pre-train encoder (B2e, MedFuse +0.014)')
print('       pre ~ col -> nguyen nhan la KIEN TRUC. Sua: bin phan vi / nut co chai token')
Db = G('t1_both') - max(G('t1_col'), G('t1_xgb'))
print(f'\nCAU 3  G(both) - max(col, xgb) = {Db:+.4f}')
print('       ~0        -> TRUNG thong tin, khac cach doc')
print('       > +0.003  -> CONG: cay bat duoc tuong tac ma `T_j = b_j + x_j*W_j` khong tao noi')
print('\nCAU 4  cau truc tu vung vi khuan (doc o NHOM NHIEM TRUNG o C11, KHONG doc macro):')
for a, b in (('t1_col_genus','t1_col_genus_shuf'), ('t1_col_sem','t1_col_sem_shuf')):
    if a in ARMS:
        print(f'       {a:<20} - {b:<20} {np.mean([g(a,s)-g(b,s) for s in SD]):+.4f}')
print('=' * 74)

print('\n=== PHAN CHUNG: tail cua t1_xgb ===')
print(f'  G(xgb) tail {Gt("t1_xgb"):+.4f}   (52a: 4-5 cot trong 40 co boi so san 0.87-2.2x,')
print('   tuc gan nhu ZERO thong tin — 109-176 ca duong => ~87-141 ca moi model fold)')
if Gt('t1_xgb') > 0.004:
    print('  !! tail nhac dang ke -> NGHI LOI, khong phai thanh tich. Kiem lai dac trung.')
else:
    print('  tail khong nhac dang ke -> DUNG du doan, ket qua doc duoc.')
ensure_res(); R.to_csv(os.path.join(RES52B, 'main_table.csv'), index=False)

=== TUYET DOI (TB qua seed) ===
                    head  mAP_anh  medium    sel   tail  tham_so
arm                                                             
mldec             0.4526   0.2369  0.1655 0.2325 0.0856  2517032
t1_col            0.4603   0.2453  0.1749 0.2388 0.0936  4341800
t1_col_zero       0.4550   0.2468  0.1718 0.2347 0.1098  4398120
t1_xgb            0.4622   0.2477  0.1751 0.2399 0.0997  4205864
t1_xgb_zero       0.4542   0.2406  0.1723 0.2348 0.0877  4216872
t1_both           0.4604   0.2475  0.1753 0.2411 0.1010  4372520
t1_col_pre        0.4387   0.2361  0.1715 0.2302 0.0907  4341800
t1_col_genus      0.4615   0.2472  0.1750 0.2383 0.0990  4375592
t1_col_genus_shuf 0.4599   0.2454  0.1738 0.2353 0.0959  4375592
t1_col_sem        0.4608   0.2473  0.1767 0.2387 0.0976  4538408
t1_col_sem_shuf   0.4607   0.2450  0.1743 0.2376 0.0927  4538408

=== G = arm - mldec (bao gom ca DUNG LUONG) ===
  t1_col               +0.0084   tail +0.0079
  t1_col_zero          +0.00

## C10 · Tráo nhân quả — bắt buộc cho mọi arm EHR

In [ ]:
ck = torch.load(os.path.join(CKPT, f's{SD[0]}.pt'), map_location=dev, weights_only=False)
MOD.load_state_dict(ck['mod']); ETK.load_state_dict(ck['etk'])
rs = np.random.default_rng(0); perm = rs.permutation(len(I_VA))
P0 = predict(MOD, ETK, I_VA); P1 = predict(MOD, ETK, I_VA, perm=I_VA[perm])
lg = lambda p: np.log(p.clip(1e-6, 1-1e-6) / (1 - p.clip(1e-6, 1-1e-6)))
print(f'=== TRAO EHR SANG BENH NHAN KHAC (s{SD[0]}) — NHAN giu nguyen ===')
print('  Doc HAI cot: corr mot minh QUA CHAT (44b tung co corr = 1.000000 ma mat 0.0056 mAP,')
print('  vi AP la chi so XEP HANG).')
for t in ARMS:
    if ARMS[t]['feat'] is None: continue
    r = float(np.corrcoef(lg(P0[t]).ravel(), lg(P1[t]).ravel())[0, 1])
    d = float(np.nanmean(ap_cols(YM[VA], P1[t])) - np.nanmean(ap_cols(YM[VA], P0[t])))
    v = ('EHR co tac dong that' if d <= -0.002 else
         'EHR KHONG duoc dung' if r > 0.9999 and d > -0.0005 else 'vung xam')
    print(f'  {t:<20} corr {r:.6f} | mAP khi trao {d:+.4f}   {v}')
    if ARMS[t].get('const'):
        assert r > 1 - 1e-9 and abs(d) < 1e-9, f'{t} la doi chung HANG SO ma trao van doi -> ro ri'
print(f'\n  moc 06b: corr = 0.999934 (H6) | cong: doi chung hang so bat bien duoi phep trao')

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

=== TRAO EHR SANG BENH NHAN KHAC (s42) — NHAN giu nguyen ===
  Doc HAI cot: corr mot minh QUA CHAT (44b tung co corr = 1.000000 ma mat 0.0056 mAP,
  vi AP la chi so XEP HANG).
  t1_col               corr 0.989508 | mAP khi trao -0.0098   EHR co tac dong that
  t1_col_zero          corr 1.000000 | mAP khi trao +0.0000   EHR KHONG duoc dung
  t1_xgb               corr 0.987727 | mAP khi trao -0.0130   EHR co tac dong that
  t1_xgb_zero          corr 1.000000 | mAP khi trao +0.0000   EHR KHONG duoc dung
  t1_both              corr 0.979887 | mAP khi trao -0.0140   EHR co tac dong that
  t1_col_pre           corr 0.916529 | mAP khi trao -0.0160   EHR co tac dong that
  t1_col_genus         corr 0.989206 | mAP khi trao -0.0107   EHR co tac dong that
  t1_col_genus_shuf    corr 0.990295 | mAP khi trao -0.0092   EHR co tac dong that
  t1_col_sem           corr 0.987583 | mAP khi trao -0.0106   EHR co tac dong that
  t1_col_sem_shuf      corr 0.990860 | mAP khi trao -0.0084   EHR co tac dong t

## C11 · Nhóm nhãn — nhiễm trùng, cấu trúc, và bội số sàn

In [ ]:
INF = [l for l in LAB40 if re.search(r'pneumon|tubercul|infect|consolidat|abscess|empyema', l, re.I)]
STR_ = [l for l in LAB40 if re.search(
    r'pneumothorax|hernia|fracture|mass|nodule|calcif|scoliosis|support device', l, re.I)]
I_INF = np.array([LAB40.index(l) for l in INF], dtype=int)
I_STR = np.array([LAB40.index(l) for l in STR_], dtype=int)
print(f'nhiem trung ({len(INF)}): {INF}\ncau truc ({len(STR_)}): {STR_}\n')
sk = f's{SD[0]}'; _r = []
for t in ARMS:
    f = os.path.join(RES52B, f'p_{t}_{sk}_ep{EPOCHS}.npy')
    if not os.path.exists(f): continue
    a = ap_cols(YM[VA], np.load(f))
    _r.append(dict(arm=t, macro=float(np.nanmean(a)),
                   nhiem_trung=float(np.nanmean(a[I_INF])), cau_truc=float(np.nanmean(a[I_STR])),
                   tail=float(np.nanmean(a[TIER['tail']]))))
T = pd.DataFrame(_r).set_index('arm')
for c in ('macro','nhiem_trung','cau_truc','tail'): T[f'd_{c}'] = T[c] - T.loc['mldec', c]
print(T.to_string(float_format=lambda v: f'{v:+.4f}'))
print('\n  EHR an nhom NHIEM TRUNG (38c col_cat +0.0186; 52a: Pneumothorax boi so san 5.04x,')
print('  Edema 2.72x — cao nhat bang, dung nhom "nhiem trung + chan thuong" cua 33 C7).')
print('  `genus`/`sem` neu co tac dung thi PHAI thay o cot `nhiem_trung`, khong phai o `macro`:')
print('  micro cham ~5.8% toan bo study (29c: 27.8% co vi khuan x 20.8% co loai ngoai top-30).')

nhiem trung (3): ['Consolidation', 'Pneumonia', 'Tuberculosis']
cau truc (10): ['Calcification of the Aorta', 'Clavicle Fracture', 'Fracture', 'Hernia', 'Hydropneumothorax', 'Mass', 'Nodule', 'Pneumothorax', 'Rib Fracture', 'Support Devices']

                    macro  nhiem_trung  cau_truc    tail  d_macro  d_nhiem_trung  d_cau_truc  d_tail
arm                                                                                                 
mldec             +0.2369      +0.2019   +0.3112 +0.0856  +0.0000        +0.0000     +0.0000 +0.0000
t1_col            +0.2453      +0.2095   +0.3227 +0.0936  +0.0084        +0.0076     +0.0115 +0.0079
t1_col_zero       +0.2468      +0.2014   +0.3257 +0.1098  +0.0099        -0.0004     +0.0145 +0.0241
t1_xgb            +0.2477      +0.2140   +0.3309 +0.0997  +0.0108        +0.0122     +0.0197 +0.0141
t1_xgb_zero       +0.2406      +0.2023   +0.3165 +0.0877  +0.0038        +0.0005     +0.0053 +0.0020
t1_both           +0.2475      +0.2074   +0.3317 

## C13 · `Tier1` có thật sự **chọn lọc** không, hay là một phép trung bình trá hình

Cả nhánh EHR đứng trên một giả định chưa ai kiểm: *mỗi patch tự hỏi EHR một câu riêng*. Nếu phân bố
attention trên 219 token EHR **gần đều** thì `Tier1` **chính là** `mean(token)`, và mọi lo ngại về
mất tính đặc hiệu áp thẳng vào đường fusion.

```
deu tuyet doi  ->  1/219 = 0.00457 moi token,  entropy = log(219) = 5.389
```

Ba con số quyết định: **entropy** (thấp = chọn lọc) · **độ lệch giữa các PATCH cùng một ảnh**
(0 = mọi vùng hỏi cùng một câu ⇒ suy biến thành FiLM toàn cục, mà `06` đã đo `≈ concat`, `p=0.308`)
· **độ lệch giữa các BỆNH NHÂN** (0 = EHR không được đọc).

Đây là loại bằng chứng cơ chế mà `06b` dùng để bác `H6` (`truyền token → logit 0.0092` so `0.25` ở
`H4`) — không phải một con số mAP.

In [ ]:
NDIAG, BSD = 256, 16
_rsd = np.random.default_rng(5)
_id = I_VA[_rsd.choice(len(I_VA), min(NDIAG, len(I_VA)), replace=False)]
_T1 = [t for t in ARMS if ARMS[t]['feat'] is not None]
print(f'{len(_id)} anh val | uniform = 1/K, entropy deu = log(K)')
print(f'{"arm":<20} {"K":>5} {"entropy":>9} {"log K":>7} {"ty le":>7} {"sd/PATCH":>9} {"sd/BN":>9}')
DIAG = {}
with torch.no_grad():
    for t in _T1:
        W = []
        for i in range(0, len(_id), BSD):
            b = next(iter(loader(_id[i:i+BSD], eval_tf, False, BSD, nw=0, drop_last=False)))
            pt = fwd(b['image'].to(dev))
            e = FEAT_T[ARMS[t]['feat']][b['erow']].to(dev)
            tok = ETK[t](e, MIDS_T[b['erow']].to(dev), MCNT_T[b['erow']].to(dev))
            w = MOD[t].t1.attn(MOD[t].pin(pt), MOD[t].ein(tok))     # [B, N_TOK, K]
            W.append(w.float().cpu())
        W = torch.cat(W); K = W.shape[-1]
        ent = float(-(W.clamp_min(1e-12) * W.clamp_min(1e-12).log()).sum(-1).mean())
        sd_patch = float(W.std(1).mean())          # lech giua cac PATCH trong cung mot anh
        sd_bn    = float(W.mean(1).std(0).mean())  # lech giua cac BENH NHAN
        DIAG[t] = dict(K=K, ent=ent, logK=math.log(K), sd_patch=sd_patch, sd_bn=sd_bn, W=W)
        print(f'{t:<20} {K:>5} {ent:>9.3f} {math.log(K):>7.3f} {ent/math.log(K):>7.1%} '
              f'{sd_patch:>9.5f} {sd_bn:>9.5f}')
_ref = DIAG.get('t1_col')
if _ref:
    _u = 1.0 / _ref['K']
    print(f'\n  doc: ty le entropy ~ 100% => attention GAN DEU => Tier1 la mean tra hinh.')
    print(f'        sd/PATCH ~ 0 => moi vung hoi CUNG mot cau => suy bien ve FiLM TOAN CUC,')
    print(f'        ma `06` da do FiLM ~ concat (p = 0.308).')
    print(f'        sd/BN ~ 0 => EHR khong duoc doc (doi chieu voi phep trao nhan qua o C10).')
    print(f'  uniform = {_u:.5f} moi token')
    # --- doi chung MIEN PHI: cac arm `_zero` dung token HANG SO, nen sd/BN cua chung la SAN cua
    #     "bien thien giua benh nhan" (chi con phan do Q=patch gay ra). Doc TY LE, dung doc so
    #     tuyet doi — mot con so nhu 0.008 khong tu no noi len gi. ---
    for _a, _b in (('t1_col','t1_col_zero'), ('t1_xgb','t1_xgb_zero')):
        if _a in DIAG and _b in DIAG:
            _r1 = DIAG[_a]['sd_bn'] / max(DIAG[_b]['sd_bn'], 1e-12)
            _r2 = DIAG[_a]['sd_patch'] / max(DIAG[_b]['sd_patch'], 1e-12)
            print(f'  {_a} / {_b}:  sd/BN {_r1:.2f}x   sd/PATCH {_r2:.2f}x   '
                  f'(token HANG SO la SAN; ~1.0x = EHR khong tao them bien thien nao)')
    # --- khoi luong attention roi vao dau: 217 cot | token TONG HOP | token BAG vi khuan ---
    m = _ref['W'].mean((0, 1))
    nC = len(COLS)
    print(f'\n  khoi luong attention (t1_col): {nC} cot {float(m[:nC].sum()):.1%} | '
          f'token TONG HOP {float(m[nC]):.1%} | token BAG vi khuan {float(m[nC+1]):.1%}')
    print(f'   (neu deu thi moi phan la {nC*_u:.1%} / {_u:.1%} / {_u:.1%})')
    FAM = [('lab', r'^lab_'), ('n_lab', r'^n_lab_'), ('dh_lab', r'^dh_lab_'), ('vital', r'^vital_'),
           ('como', r'^como_'), ('micro', r'^micro_'), ('ctx', r'^ctx_'),
           ('demo', r'^(age|gender)$'), ('mask', r'_mask$')]
    _seen = set(); print(f'\n  {"ho":<8} {"n cot":>6} {"khoi luong":>11} {"neu deu":>9} {"boi so":>7}')
    for nm, rx in FAM:
        ix = [i for i, c in enumerate(COLS) if re.search(rx, c) and i not in _seen]
        _seen |= set(ix)
        if not ix: continue
        tot_ = float(m[ix].sum()); ev = len(ix) * _u
        print(f'  {nm:<8} {len(ix):>6} {tot_:>11.2%} {ev:>9.2%} {tot_/max(ev,1e-9):>7.2f}x')
    print('  ^ boi so > 1 = ho do duoc DOC NHIEU hon muc deu. Doi chieu voi 11 nhan nhay EHR')
    print('    cua `33 C7` (nhiem trung + chan thuong) va bang boi so san cua `52a`.')
    _top = m[:nC].argsort(descending=True)[:10]
    print('\n  10 COT duoc doc nhieu nhat: ' + ' '.join(f'{COLS[int(i)]}({float(m[int(i)])/_u:.1f}x)'
          for i in _top))
for t in DIAG: DIAG[t].pop('W', None)

256 anh val | uniform = 1/K, entropy deu = log(K)
arm                      K   entropy   log K   ty le  sd/PATCH     sd/BN
t1_col                 219     1.580   5.389   29.3%   0.00374   0.00433
t1_col_zero            219     0.998   5.389   18.5%   0.00231   0.00048
t1_xgb                  42     1.421   3.738   38.0%   0.02199   0.02098
t1_xgb_zero             42     1.055   3.738   28.2%   0.01042   0.00224
t1_both                259     1.478   5.557   26.6%   0.00405   0.00323
t1_col_pre             219     1.709   5.389   31.7%   0.00561   0.00605
t1_col_genus           219     1.504   5.389   27.9%   0.00433   0.00354
t1_col_genus_shuf      219     1.786   5.389   33.1%   0.00488   0.00492
t1_col_sem             219     1.502   5.389   27.9%   0.00427   0.00352
t1_col_sem_shuf        219     1.685   5.389   31.3%   0.00392   0.00457

  doc: ty le entropy ~ 100% => attention GAN DEU => Tier1 la mean tra hinh.
        sd/PATCH ~ 0 => moi vung hoi CUNG mot cau => suy bien ve FiLM 

## C14 · Gióng hàng ở **mức logit** — hậu kỳ, 0 giây GPU

Kiến trúc không có đường nào cho query lớp `c` đọc thẳng cột XGB `c` (xem ghi chú ở `C1`). Cách duy
nhất khai thác sự gióng hàng là nối ở **mức logit** — tức **ensemble hai bộ phân loại**, đúng dạng
late fusion của MedFuse, và `48` đã đo ensembling đáng `+0.0103`.

Trọng số chọn trên **lát chọn**, báo cáo trên **val**. Chọn trên val rồi báo cáo trên val là đúng
cái bẫy §13 — và với 109–176 ca dương ở tail thì nó sẽ overfit chắc chắn.

In [ ]:
_lg = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6) / (1 - np.clip(p, 1e-6, 1-1e-6)))
Z_SL, Z_VA = _lg(XGF[SL]), _lg(XGF[VA])       # XGB: OOF-full tren SL/VA (52a)
GRID = np.r_[0.0, np.linspace(0.05, 0.6, 12)]
BASE_ARMS = [t for t in ('mldec', 't1_col', 't1_both') if t in ARMS]
print(f'{"nen":<10} {"nen mAP":>9} {"a chung":>9} {"+chung":>8} {"+theo nhan":>11}')
_res14 = []
for t in BASE_ARMS:
    f_va = os.path.join(RES52B, f'p_{t}_s{SD[0]}_ep{EPOCHS}.npy')
    f_sl = os.path.join(RES52B, f'psl_{t}_s{SD[0]}_ep{EPOCHS}.npy')
    if not (os.path.exists(f_va) and os.path.exists(f_sl)):
        print(f'{t:<10} thieu p_/psl_ -> bo qua (chay lai C8 de sinh psl_*)'); continue
    A_SL, A_VA = _lg(np.load(f_sl)), _lg(np.load(f_va))
    m0 = float(np.nanmean(ap_cols(YM[VA], A_VA)))
    # (a) mot `a` CHUNG, chon tren lat CHON
    sc = [float(np.nanmean(ap_cols(YM[SL], (1-a)*A_SL + a*Z_SL))) for a in GRID]
    a_g = float(GRID[int(np.argmax(sc))])
    m_g = float(np.nanmean(ap_cols(YM[VA], (1-a_g)*A_VA + a_g*Z_VA)))
    # (b) `a` THEO NHAN, moi nhan chon rieng tren lat CHON
    a_j = np.zeros(C)
    for j in range(C):
        s_ = [float(ap_cols(YM[SL][:, [j]], ((1-a)*A_SL[:, [j]] + a*Z_SL[:, [j]]))[0]) for a in GRID]
        a_j[j] = 0.0 if np.all(np.isnan(s_)) else GRID[int(np.nanargmax(s_))]   # nhan 0 ca duong o lat chon
    m_j = float(np.nanmean(ap_cols(YM[VA], (1-a_j)[None, :]*A_VA + a_j[None, :]*Z_VA)))
    print(f'{t:<10} {m0:>9.4f} {a_g:>9.2f} {m_g-m0:>+8.4f} {m_j-m0:>+11.4f}')
    _res14.append(dict(nen=t, mAP=m0, a_chung=a_g, d_chung=m_g-m0, d_nhan=m_j-m0,
                       n_nhan_dung_xgb=int((a_j > 0).sum())))
T14 = pd.DataFrame(_res14)
BLEND_TXT = ('—' if not len(T14) else
             f'{T14.d_chung.max():+.4f} (chung) / {T14.d_nhan.max():+.4f} (theo nhan)')
if len(T14):
    print(f'\n  so nhan chon a>0: ' + ' '.join(f'{r.nen}:{r.n_nhan_dung_xgb}/{C}'
          for r in T14.itertuples()))
    print('  DOC: `+chung > +0.005` -> ghep muc LOGIT mua duoc that, va no RE hon moi arm token-level.')
    print('       `+theo nhan` <= `+chung` -> chon theo nhan KHONG dang (47 da gap dung ket qua nay:')
    print('       cross-fit < prior-tat-ca, oracle chon 30-34/40 nhan => hieu ung RONG, khong hep).')
    print('       Ca hai ~0 -> XGB khong them gi ke ca khi duoc gion g hang thang voi dau ra.')
    ensure_res(); T14.to_csv(os.path.join(RES52B, 'blend_logit.csv'), index=False)

nen          nen mAP   a chung   +chung  +theo nhan
mldec         0.2369      0.10  +0.0015     +0.0040
t1_col        0.2453      0.05  +0.0014     +0.0024
t1_both       0.2475      0.00  +0.0000     +0.0006

  so nhan chon a>0: mldec:35/40 t1_col:32/40 t1_both:29/40
  DOC: `+chung > +0.005` -> ghep muc LOGIT mua duoc that, va no RE hon moi arm token-level.
       `+theo nhan` <= `+chung` -> chon theo nhan KHONG dang (47 da gap dung ket qua nay:
       cross-fit < prior-tat-ca, oracle chon 30-34/40 nhan => hieu ung RONG, khong hep).
       Ca hai ~0 -> XGB khong them gi ke ca khi duoc gion g hang thang voi dau ra.


## C15 · Ghi `REPORT_STAGE4_NB52B.md`

In [ ]:
L = []; w = L.append
w('# REPORT · `nb52b` — bộ mã hoá EHR có phải nút thắt, và từ vựng vi khuẩn có cấu trúc không'); w('')
w(f'`{RUN_TAG}` · backbone `ft` = `38a/{BB_RUN}` · `asl40` · `E={EPOCHS}` · **epoch cuối** · '
  f'seeds {list(SD)} · {len(ARMS)} arm **một backbone forward dùng chung** · val {int(VA.sum()):,} ảnh.')
w('')
w('| arm | tham số | mAP ảnh | tail | nhiễm trùng | `arm − mldec` |')
w('|---|---|---|---|---|---|')
for t in ARMS:
    if t not in piv.index: continue
    r = piv.loc[t]
    _i = f'{T.loc[t, "nhiem_trung"]:.4f}' if t in T.index else '—'
    w(f'| `{t}` | {NP_[t]:,} | {r["mAP_anh"]:.4f} | {r["tail"]:.4f} | {_i} | '
      + ('—' if t == 'mldec' else f'{G(t):+.4f}') + ' |')
w(''); w('```')
w(f'CAU 1  G(xgb) - G(col)            {D:+.4f}')
w(f'CAU 2  G(pre) - G(col)            {Dr:+.4f}')
w(f'CAU 3  G(both) - max(col, xgb)    {Db:+.4f}')
w(f'CAU 5  tron logit voi XGB         {globals().get(chr(66)+chr(76)+chr(69)+chr(78)+chr(68)+chr(95)+chr(84)+chr(88)+chr(84), chr(8212))}')
w('```'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **Backbone đông cứng** ⇒ EHR chỉ điều biến biểu diễn đã xong, không định hình nó. Deep '
  'conditioning (`39`, `B2d`) vẫn là cơ chế duy nhất chưa ai đóng — `11` đo `G_mid +0.0036`, gấp '
  '10× late fusion trên patch, nhưng trên cohort cũ 13 nhãn.')
w('- **`p_uni` chưa chạm.** Tỉ lệ mẫu uni-modal là thay đổi **lấy mẫu** của cả vòng lặp nên không '
  'làm được thành một arm. MedFuse đo được `+0.015` từ đúng đòn bẩy này.')
w(f'- **Cột XGB của nhãn hiếm là nhiễu.** `52a`: `Azygos Lobe 0.87×` sàn, `Clavicle Fracture 1.50×`. '
  '4–5 trong 40 cột mang gần như zero thông tin, nên `t1_xgb` **không thể** giúp tail.')
w('- **Trần cứng là nguồn nhãn.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh '
  'gần như đủ theo định nghĩa (`34`: nhãn tự động gọi thừa `Pneumonia` **5×**). Không con số nào ở '
  'đây nói gì về tiên lượng, tử vong, chuyển ICU, tái nhập viện — đó là việc của `40`.')
w(f'- **`micro` chỉ chạm ~5.8% study** (`29c`: 27.8% có vi khuẩn × 20.8% có loài ngoài top-30), nên '
  '`genus`/`sem` đọc ở **macro** là sai thang; đọc ở nhóm nhiễm trùng.')
TXT = '\n'.join(L)
ensure_res(); open(os.path.join(RES52B, 'REPORT_STAGE4_NB52B.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REPORT · `nb52b` — bộ mã hoá EHR có phải nút thắt, và từ vựng vi khuẩn có cấu trúc không

`r2` · backbone `ft` = `38a/a2_E3` · `asl40` · `E=10` · **epoch cuối** · seeds [42] · 11 arm **một backbone forward dùng chung** · val 39,293 ảnh.

| arm | tham số | mAP ảnh | tail | nhiễm trùng | `arm − mldec` |
|---|---|---|---|---|---|
| `mldec` | 2,517,032 | 0.2369 | 0.0856 | 0.2019 | — |
| `t1_col` | 4,341,800 | 0.2453 | 0.0936 | 0.2095 | +0.0084 |
| `t1_col_zero` | 4,398,120 | 0.2468 | 0.1098 | 0.2014 | +0.0099 |
| `t1_xgb` | 4,205,864 | 0.2477 | 0.0997 | 0.2140 | +0.0108 |
| `t1_xgb_zero` | 4,216,872 | 0.2406 | 0.0877 | 0.2023 | +0.0038 |
| `t1_both` | 4,372,520 | 0.2475 | 0.1010 | 0.2074 | +0.0107 |
| `t1_col_pre` | 4,341,800 | 0.2361 | 0.0907 | 0.2012 | -0.0007 |
| `t1_col_genus` | 4,375,592 | 0.2472 | 0.0990 | 0.2127 | +0.0104 |
| `t1_col_genus_shuf` | 4,375,592 | 0.2454 | 0.0959 | 0.2106 | +0.0085 |
| `t1_col_sem` | 4,538,408 | 0.2473 | 0.0976 | 0.2099 | +0.0104 |
| `t1_col_sem_shuf` 